# Setup

In [ ]:
# Working directory should be the root directory of repository.
# setwd("./")
renv::load()
source("./results/utils.R")

suppressPackageStartupMessages({
  library(CAdir)
  library(APL)
  library(SingleCellExperiment)
  library(dplyr)

  # To load the data set
  library(TENxPBMCData)
  library(Seurat)
  library(SeuratObject)
  library(scater)
  library(scuttle)
  library(scran)

  library(patchwork)
  library(ggtext)
})

options(repr.plot.width = 20, repr.plot.height = 15)

dir <- "./results/"
imgdir <- file.path(dir, "img/review/improved_vis/")
dir.create(imgdir, recursive = TRUE)

## Load data

In [ ]:
data_dir <- "./data/real/discussed/"

marker_genes <- readr::read_csv(file.path(
  data_dir,
  "tabula_muris_marker_genes.csv"
))
sce <- readRDS(file.path(
  data_dir,
  "preprocessed/tabula_muris_preprocessed_filtered.rds"
))

genevars <- modelGeneVar(sce, assay.type = "logcounts")
chosen <- getTopHVGs(genevars, n = 6000, var.threshold = NULL)

# add marker_genes
expr_markers <- marker_genes[marker_genes$gene %in% rownames(sce), ]
chosen <- c(chosen, expr_markers$gene)
chosen <- unique(chosen)
sce_sub <- sce[chosen, ]

ca <- cacomp(
  obj = logcounts(sce_sub),
  princ_coords = 3,
  dims = 30,
  top = nrow(sce_sub),
  residuals = "pearson",
  python = TRUE
)

cell_types <- sce$cell_ontology_class
cat("Number of cell types:", length(unique(cell_types)), "\n")

# CAdir

In [ ]:
set.seed(2358)
cak <- dirclust_splitmerge(
  caobj = ca,
  k = 8,
  cutoff = 55,
  method = "random",
  apl_quant = 0.99,
  counts = NULL,
  min_cells = 20,
  reps = NULL,
  make_plots = TRUE,
  qcutoff = 0.8,
  convergence_thr = 0.001
)

Annotate biclustering:

In [ ]:
cak <- annotate_biclustering(
  obj = cak,
  universe = rownames(sce_sub),
  org = "mm"
)

sce_sub$cadir <- cak@cell_clusters

cak

In [ ]:
um1 <- plotUMAP(sce_sub, colour = "cadir")
um2 <- plotUMAP(sce_sub, colour = "cell_ontology_class")

ari <- aricode::clustComp(sce_sub$cadir, sce_sub$cell_ontology_class)
cat("ARI:", ari$ARI)

um <- um1 + ggtitle(paste0("ARI: ", round(ari$ARI, 2))) + um2
um

ggsave(
  plot = um,
  filename = file.path(imgdir, "umap.png")
)

# Improved Graph plot

## graph

In [ ]:
sm <- sm_plot(
  cadir = cak,
  caobj = ca,
  rm_redund = TRUE,
  keep_end = FALSE,
  highlight_cluster = TRUE,
  show_genes = FALSE,
  annotate_clusters = FALSE,
  org = "mm",
  inlet_side = 0.12,
  title_size = 12,
  show_axis = FALSE
)

sm
ggsave(
  plot = sm,
  filename = file.path(imgdir, "tm_split_merge_plot.pdf"),
  device = cairo_pdf,
  width = 2300,
  height = 2100,
  units = "px"
)

ggsave(
  plot = sm,
  filename = file.path(imgdir, "tm_split_merge_plot.png"),
  width = 2300,
  height = 2100,
  units = "px"
)

## APLs

### large genes

In [ ]:
cls <- "B_cell"
p_bt1 <- cluster_apl(
  ca,
  cak,
  cluster = cls,
  highlight_cluster = TRUE,
  show_genes = TRUE,
  label_genes = FALSE,
  show_lines = FALSE,
  size_factor = 2,
  point_size = 1,
  ntop = 1
) +
  ggtitle("Large genes, small cells")

p_bt1

## small genes

In [ ]:
cls <- "B_cell"
p_bt2 <- cluster_apl(
  ca,
  cak,
  cluster = cls,
  highlight_cluster = TRUE,
  show_genes = TRUE,
  label_genes = FALSE,
  size_factor = 0.1,
  point_size = 3,
  ntop = 5
) +
  ggtitle("Large cells, small genes")

p_bt2

In [ ]:
cls <- "B_cell"
p_bt3 <- cluster_apl(
  ca,
  cak,
  cluster = cls,
  highlight_cluster = TRUE,
  show_genes = TRUE,
  show_cells = FALSE,
  label_genes = TRUE,
  show_lines = FALSE,
  size_factor = 1,
  point_size = 2,
  ntop = 10
) +
  ggtitle("genes only, top 10 labelled")

p_bt3

In [ ]:
cls <- "B_cell"
p_bt4 <- cluster_apl(
  caobj = ca,
  cadir = cak,
  cluster = cls,
  highlight_cluster = FALSE,
  show_genes = FALSE,
  label_genes = FALSE,
  show_lines = c("T_cell", "Immune_cell"),
  size_factor = 0.1,
  point_size = 2
) +
  ggtitle("cells only, colored by cluster")

p_bt4

## Panel

In [ ]:
library(patchwork)

fig <- (wrap_elements(sm)) +
  (p_bt1 / p_bt2) +
  plot_layout(widths = c(2, 1)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 18))
fig

ggsave(
  plot = fig,
  filename = file.path(imgdir, "new_vis_panel.png"),
  width = 3200,
  height = 1600,
  units = "px"
)

In [ ]:
fig1 <- ((p_bt4 / p_bt3) | (p_bt1 / p_bt2)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 18))
fig1


ggsave(
  plot = fig1,
  filename = file.path(imgdir, "new_vis_panel2.png"),
  width = 3200,
  height = 1600,
  units = "px"
)

### Interactive APL

In [ ]:
cls <- "B_cell"
p_bt3 <- cluster_apl(
  ca,
  cak,
  cluster = cls,
  highlight_cluster = TRUE,
  show_genes = TRUE,
  label_genes = TRUE,
  show_lines = FALSE,
  size_factor = 1,
  point_size = 2,
  ntop = Inf,
  interactive = TRUE
)

htmlwidgets::saveWidget(
  p_bt3,
  file = file.path(imgdir, "interactive_bcells.html")
)

# Session Info

In [ ]:
sessionInfo()